# Parsing Libraries

**Module 2 — Lesson 5**

The Enron corpus was originally published as plaintext RFC email -- and in that native format, Python's standard library can parse it completely with no patterns or models required. For this course we converted those emails to PDFs, so the RFC structure is lost in our extracted text.

This notebook:
1. Parses real Enron `.eml` content using `email.parser` -- showing what the parser can do on well-structured input
2. Explores the dual RFC / X-header system and its permutations
3. Shows where the parser fails on raw PDF-extracted text
4. Reconstructs RFC structure from the PDF text -- recovering ~82% field coverage with the same parser

In [1]:
import email
from email import policy
from pathlib import Path
from collections import Counter

## 1. What RFC 2822 email looks like

RFC 2822 defines a strict structure: headers as `Key: value` pairs (one per line), a blank line separator, then the body. This is what the original Enron plaintext corpus looks like -- the format `email.parser` was designed for.

The next cell embeds ~10 real Enron emails as string constants (about 230 lines). Scroll past them to the parsing code -- they're here so you can inspect the raw format and compare results.

In [2]:
# Real Enron emails embedded as string constants
# Source: Carnegie Mellon Enron corpus (public domain)

RFC_SAMPLES = {}

RFC_SAMPLES["kaminski-v/sent/1."] = """\
Message-ID: <3950956.1075856435038.JavaMail.evans@thyme>
Date: Mon, 7 May 2001 08:41:00 -0700 (PDT)
From: vince.kaminski@enron.com
To: stephen.stock@enron.com, beth.perlman@enron.com
Subject: A resume for Londom
Mime-Version: 1.0
Content-Type: text/plain; charset=us-ascii
Content-Transfer-Encoding: 7bit
X-From: Vince J Kaminski
X-To: Stephen Stock, Beth Perlman
X-cc: 
X-bcc: 
X-Folder: \\Vincent_Kaminski_Jun2001_3\\Notes Folders\\Sent
X-Origin: Kaminski-V
X-FileName: vkamins.nsf

This is a resume of one guy I met in Houston a few months ago.
He comes across as a very bright and  qualified individual. He is interested 
in
a position in London. Who is the best person in London to forward the resume 
to?

Vince
"""

RFC_SAMPLES["dasovich-j/sent/1."] = """\
Message-ID: <18782981.1075843687350.JavaMail.evans@thyme>
Date: Thu, 28 Oct 1999 03:08:00 -0700 (PDT)
From: jeff.dasovich@enron.com
To: steve.montovano@enron.com, james.steffes@enron.com, richard.shapiro@enron.com, susan.covino@enron.com, susan.mara@enron.com, mona.petrochko@enron.com, sandra.mccubbin@enron.com, bruno.gaillard@enron.com
Subject: Sempra's Marketing in Jersey
Mime-Version: 1.0
Content-Type: text/plain; charset=us-ascii
Content-Transfer-Encoding: 7bit
X-From: Jeff Dasovich
X-To: Steve Montovano, James D Steffes, Richard Shapiro, Susan M Covino, Susan J Mara, Mona L Petrochko, Sandra McCubbin, Bruno Gaillard
X-cc: 
X-bcc: 
X-Folder: \\Jeff_Dasovich_Dec2000\\Notes Folders\\Sent
X-Origin: DASOVICH-J
X-FileName: jdasovic.nsf

Will we be covering this?  The utilities in California have been very vocal about this issue.
"""

RFC_SAMPLES["allen-p/inbox/1."] = """\
Message-ID: <6788336.1075840223571.JavaMail.evans@thyme>
Date: Fri, 7 Dec 2001 10:06:42 -0800 (PST)
From: heather.dunton@enron.com
To: k..allen@enron.com
Subject: RE: West Position
Mime-Version: 1.0
Content-Type: text/plain; charset=us-ascii
Content-Transfer-Encoding: 7bit
X-From: Dunton, Heather
X-To: Allen, Phillip K.
X-cc: 
X-bcc: 
X-Folder: \\ExMerge - Allen, Phillip K.\\Inbox
X-Origin: ALLEN-P
X-FileName: phillip k allen 1-25-02.pst

Please let me know if you still need Curve Shift.

Thanks,
Heather
"""

RFC_SAMPLES["allen-p/inbox/2."] = """\
Message-ID: <6959853.1075840223786.JavaMail.evans@thyme>
Date: Mon, 10 Dec 2001 14:59:55 -0800 (PST)
From: brad.jones@enron.com
To: 
Subject: Gas P&L by day
Cc: frank.hayden@enron.com, c..gossett@enron.com
Mime-Version: 1.0
Content-Type: text/plain; charset=us-ascii
Content-Transfer-Encoding: 7bit
X-From: Jones, Brad
X-To: 
X-cc: Hayden, Frank <Frank.Hayden@ENRON.com>, Gossett, Jeffrey C. </O=ENRON/OU=NA/CN=RECIPIENTS/CN=JGOSSET>
X-bcc: 
X-Folder: \\ExMerge - Allen, Phillip K.\\Inbox
X-Origin: ALLEN-P
X-FileName: phillip k allen 1-25-02.pst

Attached is the information you have requested.

Thanks,
Brad
"""

RFC_SAMPLES["allen-p/notes_inbox/6."] = """\
Message-ID: <7048772.1075855681422.JavaMail.evans@thyme>
Date: Wed, 13 Dec 2000 07:57:00 -0800 (PST)
From: aod@newsdata.com
To: western.price.survey.contacts@ren-3.cais.net
Subject: Report on News Conference
Cc: alb@cpuc.ca.gov
Mime-Version: 1.0
Content-Type: text/plain; charset=us-ascii
Content-Transfer-Encoding: 7bit
X-From: "Arthur O'Donnell" <aod@newsdata.com>
X-To: western.price.survey.contacts@ren-3.cais.net
X-cc: alb@cpuc.ca.gov
X-bcc: 
X-Folder: \\Phillip_Allen_Jan2002_1\\Notes Folders\\Notes inbox
X-Origin: Allen-P
X-FileName: pallen.nsf

Attached is davis.doc, a quick & dirty report from today's news conference.
"""

RFC_SAMPLES["kaminski-v/sent/458."] = """\
Message-ID: <4231214.1075856436929.JavaMail.evans@thyme>
Date: Wed, 21 Mar 2001 05:29:00 -0800 (PST)
From: vince.kaminski@enron.com
To: vkaminski@aol.com
Subject: FW: meeting follow-up
Cc: bryan.seyfried@enron.com
Mime-Version: 1.0
Content-Type: text/plain; charset=us-ascii
Content-Transfer-Encoding: 7bit
X-From: Vince J Kaminski
X-To: vkaminski@aol.com
X-cc: Bryan Seyfried
X-bcc: 
X-Folder: \\Vincent_Kaminski_Jun2001_3\\Notes Folders\\Sent
X-Origin: Kaminski-V
X-FileName: vkamins.nsf

Bryan,

FYI

Vince
---------------------- Forwarded by Vince J Kaminski on 03/21/2001 01:29 PM
"""

# IMCEANOTES example -- From: is a useless routing placeholder
RFC_SAMPLES["arnold-j/deleted_items/15."] = """\
Message-ID: <25264009.1075857664798.JavaMail.evans@thyme>
Date: Thu, 29 Nov 2001 12:20:43 -0800 (PST)
From: 40enron@enron.com
To: all_enron_houston2@enron.com
Subject: Giving Season
Mime-Version: 1.0
Content-Type: text/plain; charset=us-ascii
Content-Transfer-Encoding: 7bit
X-From: Rick Buy and Mark Haedicke@ENRON <IMCEANOTES-Rick+20Buy+20and+20Mark+20Haedicke+40ENRON@ENRON.com>
X-To: All Enron Houston@ENRON
X-cc: 
X-bcc: 
X-Folder: \\JARNOLD (Non-Privileged)\\Arnold, John\\Deleted Items
X-Origin: Arnold-J
X-FileName: JARNOLD (Non-Privileged).pst

We know that these are difficult times, but the need to support our community is greater than ever.
"""


RFC_SAMPLES["lenhart-m/all_documents/3125."] = """\
Message-ID: <26351945.1075858125543.JavaMail.evans@thyme>
Date: Mon, 14 May 2001 03:42:00 -0700 (PDT)
From: derekaberle@aec.ca
To: mlenhart@enron.com
Subject: Liquidated Damages Pertaining to May 11 Gas Day
Mime-Version: 1.0
Content-Type: text/plain; charset=us-ascii
Content-Transfer-Encoding: 7bit
X-From: "Aberle, Derek" <DerekAberle@aec.ca>
X-To: "'mlenhart@enron.com'" <mlenhart@enron.com>
X-cc: 
X-bcc: 
X-Folder: \\Matthew_Lenhart_Jun2001\\Notes Folders\\All documents
X-Origin: Lenhart-M
X-FileName: mlenhar.nsf

Matt

 I am passing on these penalties for non-performance Friday for the
firm volumes on AECO at Dawn.
"""

RFC_SAMPLES["dasovich-j/inbox/1000."] = """\
Message-ID: <25321310.1075859212074.JavaMail.evans@thyme>
Date: Thu, 6 Dec 2001 09:34:56 -0800 (PST)
From: members@realmoney.com
To: members@realmoney.com
Subject: The IPO Market is Back!
Mime-Version: 1.0
Content-Type: text/plain; charset=us-ascii
Content-Transfer-Encoding: 7bit
X-From: members@realmoney.com
X-To: members@realmoney.com <prod-special@pbulk01.thestreet.com>
X-cc: 
X-bcc: 
X-Folder: \\Jeff_Dasovich_Jan2002\\Dasovich, Jeff\\Inbox
X-Origin: Dasovich-J
X-FileName: jdasovic (Non-Privileged).pst

***The IPO market is back!***
"""


RFC_SAMPLES["parks-j/deleted_items/779."] = """\
Message-ID: <29991894.1075841375652.JavaMail.evans@thyme>
Date: Wed, 6 Feb 2002 19:03:13 -0800 (PST)
From: ed.mcmichael@enron.com
To: louis.dicarlo@enron.com
Subject: RE: Entex Early Termination Document
Cc: joe.parks@enron.com, kay.mann@enron.com, chip.schneider@enron.com
Mime-Version: 1.0
Content-Type: text/plain; charset=us-ascii
Content-Transfer-Encoding: 7bit
X-From: McMichael Jr., Ed </O=ENRON/OU=NA/CN=RECIPIENTS/CN=EMCMICH>
X-To: Dicarlo, Louis </O=ENRON/OU=NA/CN=RECIPIENTS/CN=Ldicarlo>
X-cc: Parks, Joe </O=ENRON/OU=NA/CN=RECIPIENTS/CN=Jparks>, Mann, Kay </O=ENRON/OU=NA/CN=RECIPIENTS/CN=Kmann>, Schneider, Chip </O=ENRON/OU=NA/CN=RECIPIENTS/CN=Cschneid>
X-bcc: 
X-Folder: \\Joe_Parks_Jan2002\\Parks, Joe\\Deleted Items
X-Origin: Parks-J
X-FileName: jparks (Non-Privileged).pst

Louis,
The dash looks good.  I made a few edits and have highlighted (in yellow) the items that still need input.
"""

# Show the first sample
raw_eml = RFC_SAMPLES["kaminski-v/sent/1."]
print(raw_eml)

Message-ID: <3950956.1075856435038.JavaMail.evans@thyme>
Date: Mon, 7 May 2001 08:41:00 -0700 (PDT)
From: vince.kaminski@enron.com
To: stephen.stock@enron.com, beth.perlman@enron.com
Subject: A resume for Londom
Mime-Version: 1.0
Content-Type: text/plain; charset=us-ascii
Content-Transfer-Encoding: 7bit
X-From: Vince J Kaminski
X-To: Stephen Stock, Beth Perlman
X-cc: 
X-bcc: 
X-Folder: \Vincent_Kaminski_Jun2001_3\Notes Folders\Sent
X-Origin: Kaminski-V
X-FileName: vkamins.nsf

This is a resume of one guy I met in Houston a few months ago.
He comes across as a very bright and  qualified individual. He is interested 
in
a position in London. Who is the best person in London to forward the resume 
to?

Vince



In [3]:
# A single function call parses the entire email
# policy.default enables modern parsing with proper Unicode handling
msg = email.message_from_string(raw_eml, policy=policy.default)

# Direct field access — no patterns, no models
print(f"From:    {msg['From']}")
print(f"To:      {msg['To']}")
print(f"Date:    {msg['Date']}")
print(f"Subject: {msg['Subject']}")
print()

# get_body() handles multipart MIME automatically
body_part = msg.get_body(preferencelist=("plain",))
body = body_part.get_content().strip() if body_part else ""
print(f"Body:    {body!r}")

From:    vince.kaminski@enron.com
To:      stephen.stock@enron.com, beth.perlman@enron.com
Date:    Mon, 07 May 2001 08:41:00 -0700
Subject: A resume for Londom

Body:    'This is a resume of one guy I met in Houston a few months ago.\nHe comes across as a very bright and  qualified individual. He is interested \nin\na position in London. Who is the best person in London to forward the resume \nto?\n\nVince'


Notice: every field is populated with a single dictionary-style lookup — `msg["From"]`, `msg["Subject"]`. No patterns, no models, no parsing logic. The structure was already there in the file.

## 2. Wrapping the Parser

The `parse_eml` function wraps the string-based parser into a structured dictionary -- the pattern you'd use in a real pipeline. (In a file-based pipeline you'd read from files with `email.message_from_file` -- here we use `message_from_string` since the samples are embedded in the notebook.)

In [4]:
def parse_eml(raw_text):
    """Parse an RFC 2822 email string into a structured dict."""
    msg = email.message_from_string(raw_text, policy=policy.default)

    # get_body handles multipart MIME — returns the plain-text part
    body_part = msg.get_body(preferencelist=("plain",))
    body = body_part.get_content() if body_part else ""

    return {
        "from":    msg["From"],
        "to":      msg["To"],
        "cc":      msg["Cc"],
        "date":    msg["Date"],
        "subject": msg["Subject"],
        "body":    body.strip(),
    }

In [5]:
# Call parse_eml on the first sample
record = parse_eml(RFC_SAMPLES["kaminski-v/sent/1."])

for field, value in record.items():
    if field == "body":
        print(f"  body:    {str(value)[:80]!r}")
    else:
        print(f"  {field:<10} {str(value)!r}")

  from       'vince.kaminski@enron.com'
  to         'stephen.stock@enron.com, beth.perlman@enron.com'
  cc         'None'
  date       'Mon, 07 May 2001 08:41:00 -0700'
  subject    'A resume for Londom'
  body:    'This is a resume of one guy I met in Houston a few months ago.\nHe comes across a'


## 3. The dual header system

The Enron corpus carries two parallel header systems. `From` and `To` are the standard RFC headers. `X-From` and `X-To` were added by the Lotus Notes/Exchange relay as the emails were routed.

On investigation, you will find that these headers often contain different types of information in each email. Sometimes the main header will include an email and a display name, while the Lotus header will contain only a display name. Sometimes it's the opposite -- and sometimes both will carry both types (email and name), but one will be missing an name or email pair entirely.

The examples below show five real permutations from the corpus.

In [6]:
# Parse embedded samples to show all five From / X-From permutations

permutation_labels = {
    "kaminski-v/sent/1.":            "A: From=bare email, X-From=name only",
    "parks-j/deleted_items/779.":    "B: From=bare email, X-From=Name <DN path>",
    "lenhart-m/all_documents/3125.": "C: From=bare email, X-From=Name <email> (external)",
    "dasovich-j/inbox/1000.":        "D: Both bare email (mailing list)",
    "arnold-j/deleted_items/15.":    "E: IMCEANOTES in X-From",
}

for key, label in permutation_labels.items():
    raw = RFC_SAMPLES[key]
    msg = email.message_from_string(raw, policy=policy.default)

    print(f"=== {label} ===")
    print(f"  From:   {msg['From']}")
    print(f"  X-From: {str(msg['X-From'])[:70]}")
    print(f"  To:     {str(msg['To'])[:60]}")
    print(f"  X-To:   {str(msg['X-To'])[:60]}")
    print()

=== A: From=bare email, X-From=name only ===
  From:   vince.kaminski@enron.com
  X-From: Vince J Kaminski
  To:     stephen.stock@enron.com, beth.perlman@enron.com
  X-To:   Stephen Stock, Beth Perlman

=== B: From=bare email, X-From=Name <DN path> ===
  From:   ed.mcmichael@enron.com
  X-From: McMichael Jr., Ed </O=ENRON/OU=NA/CN=RECIPIENTS/CN=EMCMICH>
  To:     louis.dicarlo@enron.com
  X-To:   Dicarlo, Louis </O=ENRON/OU=NA/CN=RECIPIENTS/CN=Ldicarlo>

=== C: From=bare email, X-From=Name <email> (external) ===
  From:   derekaberle@aec.ca
  X-From: "Aberle, Derek" <DerekAberle@aec.ca>
  To:     mlenhart@enron.com
  X-To:   "'mlenhart@enron.com'" <mlenhart@enron.com>

=== D: Both bare email (mailing list) ===
  From:   members@realmoney.com
  X-From: members@realmoney.com
  To:     members@realmoney.com
  X-To:   members@realmoney.com <prod-special@pbulk01.thestreet.com>

=== E: IMCEANOTES in X-From ===
  From:   40enron@enron.com
  X-From: Rick Buy and Mark Haedicke@ENRON <IMCEANOTE

Five permutations from the real corpus:

- **A** (bare email + name only): `From` has the email, `X-From` has the name. The most common pattern (~60%).
- **B** (bare email + DN path): `From` has a bare email, `X-From` has a Lotus Notes Distinguished Name (`McMichael Jr., Ed </O=ENRON/...>`). The name is extractable but the path is not a usable email address.
- **C** (bare email + Name <email>): `From` has a bare email, `X-From` has both name and email in `"Name" <email>` format. The X-header is strictly more informative.
- **D** (both bare email): Neither header carries a display name. Common with mailing lists and automated senders.
- **E** (IMCEANOTES): `X-From` contains an encoded identity that needs URL-decoding. `From` may have a real email or a placeholder.

Building a complete sender record from these fragments is a combining problem -- one we'll explore in the next notebook. For now, `email.parser` gives you clean access to both headers. The raw material is there.

While these inconsistencies may be unique to this dataset -- inconsistency itself is a consistent feature of most datasets. Keep this in mind when doing this for yourself. 

## 4. Coverage on RFC-format email

The parser works on individual emails. Before moving on, run it across all the notebook samples to see how consistently it extracts fields across different senders, folders, and header combinations.

The cell below loads all notebook samples into a list and prints a summary of what's available. Run it to see the sample set.

In [7]:
# Use all embedded samples -- different folders, senders, field combinations
# Keys serve as labels matching the original Enron Maildir paths
sample_rfc_emails = list(RFC_SAMPLES.values())
sample_rfc_labels = list(RFC_SAMPLES.keys())

print(f"Loaded {len(sample_rfc_emails)} real Enron emails (embedded samples)")
print()
for label, raw in zip(sample_rfc_labels, sample_rfc_emails):
    msg = email.message_from_string(raw, policy=policy.default)
    print(f"  {label}")
    print(f"    From:    {msg['From']}")
    print(f"    Subject: {msg['Subject']}")
    print()

Loaded 10 real Enron emails (embedded samples)

  kaminski-v/sent/1.
    From:    vince.kaminski@enron.com
    Subject: A resume for Londom

  dasovich-j/sent/1.
    From:    jeff.dasovich@enron.com
    Subject: Sempra's Marketing in Jersey

  allen-p/inbox/1.
    From:    heather.dunton@enron.com
    Subject: RE: West Position

  allen-p/inbox/2.
    From:    brad.jones@enron.com
    Subject: Gas P&L by day

  allen-p/notes_inbox/6.
    From:    aod@newsdata.com
    Subject: Report on News Conference

  kaminski-v/sent/458.
    From:    vince.kaminski@enron.com
    Subject: FW: meeting follow-up

  arnold-j/deleted_items/15.
    From:    40enron@enron.com
    Subject: Giving Season

  lenhart-m/all_documents/3125.
    From:    derekaberle@aec.ca
    Subject: Liquidated Damages Pertaining to May 11 Gas Day

  dasovich-j/inbox/1000.
    From:    members@realmoney.com
    Subject: The IPO Market is Back!

  parks-j/deleted_items/779.
    From:    ed.mcmichael@enron.com
    Subject: RE: E

Now parse every sample and count which fields are present. This loops through each email, extracts the standard fields (`From`, `To`, `Cc`, `Date`, `Subject`), and tallies how many samples contain each one.

Run the cell to see the coverage breakdown.

In [8]:
# Parse all samples and measure field coverage
fields = ["From", "To", "Cc", "Date", "Subject"]
field_counts = Counter()
records = []

for raw in sample_rfc_emails:
    msg = email.message_from_string(raw, policy=policy.default)
    body_part = msg.get_body(preferencelist=("plain",))
    body = body_part.get_content().strip() if body_part else ""

    record = {
        "from":    msg["From"],
        "to":      msg["To"],
        "cc":      msg["Cc"],
        "date":    msg["Date"],
        "subject": msg["Subject"],
        "body":    body,
    }
    records.append(record)

    for f in fields:
        if msg[f] is not None:
            field_counts[f] += 1

n = len(sample_rfc_emails)
print(f"Field coverage across {n} RFC-format emails:")
print()
for f in fields:
    count = field_counts[f]
    pct = count / n * 100
    bar = "\u2588" * int(pct / 5)
    print(f"  {f:<10} {count}/{n}  {pct:5.1f}%  {bar}")

Field coverage across 10 RFC-format emails:

  From       10/10  100.0%  ████████████████████
  To         10/10  100.0%  ████████████████████
  Cc         4/10   40.0%  ████████
  Date       10/10  100.0%  ████████████████████
  Subject    10/10  100.0%  ████████████████████


The coverage numbers tell you which fields you can reliably depend on. Now look at the actual parsed records -- what does the extracted data look like for each email?

Run the cell below to print every record. Check the `from`, `to`, and `subject` fields. Notice that `cc` is `None` for emails that had no CC recipients -- that's expected, not an error.

In [9]:
# Print parsed records — every field populated where the sender included it
for i, rec in enumerate(records, 1):
    print(f"=== Email {i} ===")
    for field, value in rec.items():
        if field == "body":
            print(f"  body:    {value[:60]!r}")
        else:
            print(f"  {field:<10} {str(value)!r}")
    print()

print("Cc is None for emails that had no CC recipients — this is correct, not a parsing failure.")

=== Email 1 ===
  from       'vince.kaminski@enron.com'
  to         'stephen.stock@enron.com, beth.perlman@enron.com'
  cc         'None'
  date       'Mon, 07 May 2001 08:41:00 -0700'
  subject    'A resume for Londom'
  body:    'This is a resume of one guy I met in Houston a few months ag'

=== Email 2 ===
  from       'jeff.dasovich@enron.com'
  to         'steve.montovano@enron.com, james.steffes@enron.com, richard.shapiro@enron.com, susan.covino@enron.com, susan.mara@enron.com, mona.petrochko@enron.com, sandra.mccubbin@enron.com, bruno.gaillard@enron.com'
  cc         'None'
  date       'Thu, 28 Oct 1999 03:08:00 -0700'
  subject    "Sempra's Marketing in Jersey"
  body:    'Will we be covering this?  The utilities in California have '

=== Email 3 ===
  from       'heather.dunton@enron.com'
  to         'k..allen@enron.com'
  cc         'None'
  date       'Fri, 07 Dec 2001 10:06:42 -0800'
  subject    'RE: West Position'
  body:    'Please let me know if you still need Curve 

The records above show field values as raw strings. With `policy.default`, the `From` and `To` headers expose an `.addresses` attribute that splits recipients into structured objects with `.display_name` and `.addr_spec`.

In many email datasets, `From:` carries `"Name" <email>` format and `.addresses` returns both. In this corpus, almost every `From:` line is a bare email address -- display names live in the X-headers instead.

Run the cell below to see what `.addresses` gives you on these samples, and compare it to `X-From`.

In [10]:
# .addresses splits recipients into structured objects — but only from what's in the header

for i, raw in enumerate(sample_rfc_emails[:3], 1):
    msg = email.message_from_string(raw, policy=policy.default)

    print(f"Email {i}:")

    # RFC From header — .addresses gives structured access
    if msg["From"]:
        for addr in msg["From"].addresses:
            print(f"  From .addresses: name={addr.display_name!r}, email={addr.addr_spec!r}")

    # X-From header — the names live here, but it's just a string
    if msg["X-From"]:
        print(f"  X-From (string): {msg['X-From']}")

    print()

Email 1:
  From .addresses: name='', email='vince.kaminski@enron.com'
  X-From (string): Vince J Kaminski

Email 2:
  From .addresses: name='', email='jeff.dasovich@enron.com'
  X-From (string): Jeff Dasovich

Email 3:
  From .addresses: name='', email='heather.dunton@enron.com'
  X-From (string): Dunton, Heather



In this corpus, `.display_name` is empty for every `From` address -- the Enron relay system stored bare email addresses in `From:` and put display names in `X-From:` instead. This is specific to how the Enron Exchange/Lotus Notes system worked, not a universal property of email.

If you're working with a different dataset (e.g., Gmail exports, Outlook .pst files), `From:` will often carry `"Name" <email>` format and `.addresses` will return both fields populated. The dual header system and its combining challenges are particular to this corpus.

Your corpus will likely have its own particulars.

## 4b. Address utilities

The `email` module also includes `email.utils` — a set of lower-level utilities for working with addresses and dates. Two are worth knowing for later, when you need to split flat recipient strings into structured records for the graph.

- `email.utils.parseaddr()` splits a single `"Name" <email>` string into a `(name, email)` tuple
- `email.utils.getaddresses()` splits a comma-separated recipient list into a list of `(name, email)` tuples, correctly handling commas inside display names

Run the cell below to see how they handle the Enron address formats.

In [11]:
from email.utils import parseaddr, getaddresses

# parseaddr: splits "Name <email>" into (name, email)
print("parseaddr examples:")
print(f"  {parseaddr('"Aberle, Derek" <DerekAberle@aec.ca>')}")
print(f"  {parseaddr('jim.schwieger@enron.com')}")
print(f"  {parseaddr('McMichael Jr., Ed </O=ENRON/OU=NA/CN=RECIPIENTS/CN=EMCMICH>')}")
print()

# getaddresses: handles comma-separated lists with names that contain commas
print("getaddresses examples:")
recip_list = '"Lay, Kenneth" <klay@enron.com>, "Whalley, Greg" <greg.whalley@enron.com>'
for name, addr in getaddresses([recip_list]):
    print(f"  name={name!r}, email={addr!r}")


parseaddr examples:
  ('Aberle, Derek', 'DerekAberle@aec.ca')
  ('', 'jim.schwieger@enron.com')
  ('', '')

getaddresses examples:
  name='Lay, Kenneth', email='klay@enron.com'
  name='Whalley, Greg', email='greg.whalley@enron.com'


`parseaddr` handles the standard `"Name" <email>` format well. For Enron-specific formats like Exchange DN paths, it extracts what it can — the name portion — even if the "email" part is an X.500 path rather than an address.

These utilities won't solve every parsing problem in this corpus, but they're the standard tools for the job and handle many edge cases (quoted commas, nested angle brackets, encoded names) that manual string splitting would miss.

## 5. The boundary: raw PDF text

Now try `email.parser` on the PDF-extracted `.txt` files from Module 1 — without any transformation. Run the cells below to load the files, inspect one, attempt to parse it, and see the result.

In [12]:
TXT_DIR = Path("data/extracted_text")
txt_files = sorted(TXT_DIR.glob("*.txt"))
print(f"Found {len(txt_files):,} extracted text files")

Found 4,911 extracted text files


In [13]:
# Inspect a PDF-extracted email
if txt_files:
    sample_text = txt_files[1].read_text(encoding="utf-8")

    print("=== PDF-extracted text (first 400 chars) ===")
    print(sample_text[:400])
    print()
    print("Notice: 'From:' is on its own line; the sender name is on the next line.")
    print("There are no 'Key: value' pairs — just a sequence of lines.")

=== PDF-extracted text (first 400 chars) ===
CONFIDENTIAL
Enron Corp.
Case No. EC-2002-01038
Doc No. E0000D1D0
Date: 01/15/2003
ENRON CORP. - PRODUCED PURSUANT TO FERC SUBPOENA.
SUBJECT TO PROTECTIVE ORDER. CONFIDENTIAL TREATMENT REQUESTED.
CONFIDENTIAL - SUBJECT TO PROTECTIVE ORDER
From:
EDIS Email Service <edismail@incident.com>
Sent:
Fri, 5 Apr 2002 00:07:00 -0800 (PST)
To:
Motley, Matt <matt.motley@enron.com>
Subject:
[EDIS]  EQ 4 2 SAN 

Notice: 'From:' is on its own line; the sender name is on the next line.
There are no 'Key: value' pairs — just a sequence of lines.


In [14]:
# Try email.parser on the PDF-extracted text
if txt_files:
    msg = email.message_from_string(sample_text, policy=policy.default)

    print("=== email.parser result on PDF-extracted text ===")
    print(f"  From:    {msg['From']!r}")
    print(f"  To:      {msg['To']!r}")
    print(f"  Date:    {msg['Date']!r}")
    print(f"  Subject: {msg['Subject']!r}")

    body_part = msg.get_body(preferencelist=("plain",))
    body = body_part.get_content().strip() if body_part else ""
    print(f"  Body:    {body[:80]!r}")

    print()
    print("All header fields return None.")
    print("The RFC structure is absent from PDF-extracted text.")

=== email.parser result on PDF-extracted text ===
  From:    None
  To:      None
  Date:    None
  Subject: None
  Body:    ''

All header fields return None.
The RFC structure is absent from PDF-extracted text.


Every field is `None`. The RFC structure doesn't survive PDF extraction — `email.parser` receives lines of text, not `Key: value` pairs, and returns an empty object.

But the information IS there. Look at the raw text: `From:` on one line, the sender name on the next, `Sent:` on one line, the date on the next. The structure is recoverable — it just needs reassembly.

In [15]:
# Measure the failure rate across a larger sample of PDF-extracted files

SAMPLE_SIZE = min(200, len(txt_files))
sample_files = txt_files[:SAMPLE_SIZE]

pdf_field_counts = Counter()
total_parsed = 0

for path in sample_files:
    try:
        text = path.read_text(encoding="utf-8")
        msg = email.message_from_string(text, policy=policy.default)
        for f in ["From", "To", "Date", "Subject"]:
            if msg[f] is not None:
                pdf_field_counts[f] += 1
        total_parsed += 1
    except Exception:
        pass

print(f"email.parser field coverage on {total_parsed} PDF-extracted files:")
print()
for f in ["From", "To", "Date", "Subject"]:
    count = pdf_field_counts[f]
    pct = count / total_parsed * 100 if total_parsed else 0
    bar = "\u2588" * int(pct / 5) if count else "(nothing)"
    print(f"  {f:<10} {count}/{total_parsed}  {pct:5.1f}%  {bar}")

email.parser field coverage on 200 PDF-extracted files:

  From       0/200    0.0%  (nothing)
  To         0/200    0.0%  (nothing)
  Date       0/200    0.0%  (nothing)
  Subject    0/200    0.0%  (nothing)


## 6. Reconstructing RFC structure

The parser returns nothing because the PDF text doesn't have `Key: value` pairs on single lines. But the information is there — `From:` on one line, the sender on the next, `Sent:` on one line, the date on the next.

If we strip the boilerplate and reassemble these label/value pairs onto single lines, we can reconstruct something close enough to RFC format for `email.parser` to read.

The function below does this in two passes:
1. Remove lines that match known boilerplate patterns
2. Walk through what's left, joining each label with its value onto one line. After `Subject:`, everything else is body.

This function uses regex (regular expressions) to reconstruct the emails. We'll cover that in more detail in the next lesson. 

For now, just focus on what it does, rather than how it does it.

In [16]:
import re

BOILERPLATE_RE = re.compile(
    r'^(?:CONFIDENTIAL|UNCLASSIFIED|Enron Corp|Case No|Doc No|Date:\s*\d{2}/\d{2}/\d{4}|'
    r'ENRON CORP|SUBJECT TO|RELEASE IN|ENRON-\d|COMPIBENT|Prima Corp|'
    r'Bate:|Bane:|Daze:|Dare:|Boo No|Boor No|ENRON COR[PE]|'
    r'CONFIDENTIAL\s*-)',
    re.IGNORECASE
)

HEADER_LABELS = ["From:", "Sent:", "To:", "Cc:", "Bcc:", "Subject:"]
HEADER_SET = set(HEADER_LABELS)


def reconstruct_rfc(text):
    """Reconstruct RFC-style headers from PDF-extracted text.

    Strips boilerplate, then reassembles label/value pairs onto single lines.
    Maps 'Sent:' to 'Date:' for email.parser compatibility.
    """
    lines = text.split("\n")
    cleaned = [l.strip() for l in lines if l.strip() and not BOILERPLATE_RE.match(l.strip())]

    rfc_lines = []
    body_lines = []
    i = 0
    in_headers = True

    while i < len(cleaned):
        line = cleaned[i]

        if not in_headers:
            body_lines.append(line)
            i += 1
            continue

        if line in HEADER_SET:
            label = line
            values = []
            i += 1
            while i < len(cleaned) and cleaned[i] not in HEADER_SET:
                values.append(cleaned[i])
                i += 1
                # Subject is always the last header — one line of value, then body
                if label == "Subject:" and len(values) == 1:
                    in_headers = False
                    break
            rfc_lines.append(f"{label} {', '.join(values)}" if values else label)

        elif any(line.startswith(l) for l in HEADER_LABELS):
            rfc_lines.append(line)
            if line.startswith("Subject:"):
                in_headers = False
            i += 1

        else:
            in_headers = False
            body_lines.append(line)
            i += 1

    while i < len(cleaned):
        body_lines.append(cleaned[i])
        i += 1

    rfc_text = "\n".join(rfc_lines) + "\n\n" + "\n".join(body_lines)
    return re.sub(r'^Sent:', 'Date:', rfc_text, flags=re.MULTILINE)

Run the cell below to reconstruct a single PDF-extracted file and parse it. Compare the result to the `None` values from section 5.

In [17]:
# Reconstruct and parse a single PDF-extracted file
sample_text = txt_files[0].read_text(encoding="utf-8")
rfc_text = reconstruct_rfc(sample_text)

print("=== Reconstructed text (first 10 lines) ===")
for line in rfc_text.split("\n")[:10]:
    print(f"  {line[:80]}")

print("\n=== Parsed result ===")
msg = email.message_from_string(rfc_text, policy=policy.default)
print(f"  From:    {msg['From']}")
print(f"  Date:    {msg['Date']}")
print(f"  To:      {str(msg['To'])[:60]}")
print(f"  Subject: {str(msg['Subject'])[:60]}")

body_part = msg.get_body(preferencelist=("plain",))
body = body_part.get_content().strip() if body_part else ""
print(f"  Body:    {body[:80]!r}")

=== Reconstructed text (first 10 lines) ===
  From: Rob Bradley <rob.bradley@enron.com>
  Date: Tue, 21 Nov 2000 08:34:00 -0800 (PST)
  To: Kenneth Lay <kenneth.lay@enron.com>
  Cc: Alhamd Alkhayat <alhamd.alkhayat@enron.com>
  Subject: Remarks to EES Employees--December 1
  
  [REDACTED] B6
  EES employee meeting next Friday, and I put together the attached as a
  potential framework.  It provides an Enron problem list of 1985/86 versus
  today and a brief look at our different visions.

=== Parsed result ===
  From:    Rob Bradley <rob.bradley@enron.com>
  Date:    Tue, 21 Nov 2000 08:34:00 -0800
  To:      Kenneth Lay <kenneth.lay@enron.com>
  Subject: Remarks to EES Employees--December 1
  Body:    '[REDACTED] B6\nEES employee meeting next Friday, and I put together the attached '


Fields that were `None` seconds ago are now populated -- same parser, same PDF text, just restructured.

One limitation: the reconstruction joins multi-line To/Cc values with commas (`', '.join(values)`). If a name already contains a comma (`McMichael Jr., Ed`), the result is ambiguous -- `getaddresses` can't tell where one recipient ends and the next begins. This affects multi-recipient emails and is one reason we move to templates in the next notebook.

Now run the reconstruction across the full corpus and measure coverage.

In [18]:
# Coverage: reconstruction vs raw parsing across the full corpus
field_counts_recon = Counter()
field_counts_raw = Counter()
errors = 0

for f in txt_files:
    text = f.read_text(encoding="utf-8")

    # With reconstruction
    rfc = reconstruct_rfc(text)
    msg = email.message_from_string(rfc, policy=policy.default)
    for field in ["From", "Date", "To", "Subject"]:
        try:
            if msg[field]:
                field_counts_recon[field] += 1
        except Exception:
            errors += 1

    # Without reconstruction (raw)
    msg_raw = email.message_from_string(text, policy=policy.default)
    for field in ["From", "Date", "To", "Subject"]:
        try:
            if msg_raw[field]:
                field_counts_raw[field] += 1
        except Exception:
            pass

total = len(txt_files)
print(f"{'Field':<10} {'Raw PDF':>10} {'Reconstructed':>15}")
print("-" * 38)
for field in ["From", "Date", "To", "Subject"]:
    raw = field_counts_raw[field]
    recon = field_counts_recon[field]
    print(f"{field:<10} {raw:>9,} {recon:>14,}  ({recon/total*100:.1f}%)")

print(f"\n({errors} parse errors on edge-case addresses)")

Field         Raw PDF   Reconstructed
--------------------------------------
From               0          4,011  (81.7%)
Date               0          4,012  (81.7%)
To                 0          3,920  (79.8%)
Subject            0          3,979  (81.0%)

(12 parse errors on edge-case addresses)


From 0% to ~82% coverage on the same files, with the same parser -- the only difference is restructuring the text before parsing.

The remaining ~18% are files where the boilerplate patterns or header layout don't match the reconstruction assumptions -- OCR-damaged headers, unusual formatting, or multi-page chains where the structure breaks down. The reconstruction also assumes Subject values are exactly one line -- if a subject wraps, only the first line is captured. This is a reasonable tradeoff for most emails, but contributes to the failure rate.

## 7. Side by side: RFC vs PDF

The same email content, two very different parser outcomes -- entirely dependent on whether the RFC structure is present.

In [19]:
rfc_version = """Message-ID: <6788336.1075840223571.JavaMail.evans@thyme>
Date: Thu, 18 Oct 2001 08:02:00 -0700 (PDT)
From: matthew.lenhart@enron.com
To: val.generes@enron.com
Subject: RE: Tuesday Dinner
Mime-Version: 1.0
Content-Type: text/plain; charset=us-ascii

Thanks - see you then."""

pdf_extracted = """CONFIDENTIAL
Enron Corp.
Case No. EC-2002-01038
Doc No. E0048ADF3
Date: 01/15/2003
ENRON CORP. - PRODUCED PURSUANT TO FERC SUBPOENA.
SUBJECT TO PROTECTIVE ORDER. CONFIDENTIAL TREATMENT REQUESTED.
RELEASE IN PART
From:
Matthew Lenhart
Sent:
Thursday, October 18, 2001
To:
val.generes@ac.com @ ENRO
Subject:
RE: Tuesday Dinner

Thanks - see you then."""

print("=== RFC 2822 email — email.parser result ===")
rfc_msg = email.message_from_string(rfc_version, policy=policy.default)
print(f"  From:    {rfc_msg['From']!r}")
print(f"  To:      {rfc_msg['To']!r}")
print(f"  Date:    {rfc_msg['Date']!r}")
print(f"  Subject: {rfc_msg['Subject']!r}")
print()

print("=== PDF-extracted text — email.parser result ===")
pdf_msg = email.message_from_string(pdf_extracted, policy=policy.default)
print(f"  From:    {pdf_msg['From']!r}")
print(f"  To:      {pdf_msg['To']!r}")
print(f"  Date:    {pdf_msg['Date']!r}")
print(f"  Subject: {pdf_msg['Subject']!r}")
print()
print("Same email, same content — completely different parser output.")
print("The RFC structure is what makes the library work.")

=== RFC 2822 email — email.parser result ===
  From:    'matthew.lenhart@enron.com'
  To:      'val.generes@enron.com'
  Date:    'Thu, 18 Oct 2001 08:02:00 -0700'
  Subject: 'RE: Tuesday Dinner'

=== PDF-extracted text — email.parser result ===
  From:    None
  To:      None
  Date:    None
  Subject: None

Same email, same content — completely different parser output.
The RFC structure is what makes the library work.


## 8. Other libraries

Python's `email` module handles RFC 2822 email well. Other libraries solve specific edge cases but share the same fundamental constraint.

| Library | Strength | Constraint |
|---|---|---|
| `email` (stdlib) | Complete RFC 2822 + MIME support. No dependencies. | Address parsing with comma-names + semicolons can be ambiguous |
| `email.utils` (stdlib) | `parseaddr` and `getaddresses` split address strings into `(name, email)` tuples. Handles quoted commas and nested brackets. | Works on individual address strings, not full emails |
| `python-dateutil` | `dateutil.parser.parse()` handles free-form dates like `"Monday, June 26, 2000 05:20 AM"`. | Not an email library — but the standard tool for the date-parsing step when preparing records for import |
| `mailparser` | Auto date normalization, address splitting, attachment metadata | Same RFC input requirement |
| `flanker` | Handles RFC edge cases and MIME quirks; strong address parser | Less actively maintained; Python 2 heritage |
| `pyzmail36` | Decodes encoded headers (`=?UTF-8?B?...?=`) and charset conversion | Mostly useful for international email |

The email-specific libraries all share the same constraint: RFC-format input required. `email.utils` and `dateutil` work on individual strings and are useful regardless of input format.

## 9. When to use a parsing library

**Use a parsing library when:**
- You have raw `.eml` or `.mbox` files
- Your corpus comes from Gmail, Outlook, or any standard email client export
- Your emails haven't been through a PDF conversion or other transformation

**It faces its limits when:**
- Your emails were extracted from PDFs
- Headers span multiple columns or were printed-to-PDF or scanned

If you are looking at your corpus and find that it could fit cleanly into RFC structure, and the scans are of a high-enough quality -- reconstruction and parsing could be the fastest path to graph.

In the next lesson, you'll learn how to use regex entirely alone to define patterns and parse emails, or use it in combination with the reconstruction we just did to handle only edge cases.

## Summary

- `email.message_from_string()` parses RFC 2822 email completely -- headers, body, MIME -- with a single function call
- The Enron corpus carries two parallel header systems (RFC + Lotus Notes X-headers), each with different information -- neither is reliably complete on its own
- Five real permutations show the range: bare email + name, bare + Name <email>, bare + DN path, both bare, and IMCEANOTES
- `email.utils.parseaddr` and `getaddresses` split address strings into `(name, email)` tuples -- the standard tools for the recipient-splitting problem you'll encounter when preparing records for import
- `python-dateutil` parses free-form date strings into `datetime` objects -- the standard tool for date normalization
- On raw PDF-extracted text, the parser returns nothing -- the RFC structure doesn't survive the PDF conversion
- **Reconstructing** the RFC structure from the PDF text -- stripping boilerplate and reassembling label/value pairs -- recovers ~82% field coverage with the same parser
- The remaining ~18% need other approaches -- the next notebooks explore those
